### Tworzenie modelu regresji liniowej

In [15]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

df = pd.read_csv('dane_gotowe_ML.csv')

df.head()

,Cena,Przebieg,Pojemność skokowa,Moc,Wiek,Rodzaj paliwa_Benzyna+LPG,Rodzaj paliwa_Diesel,Rodzaj paliwa_Elektryczny,Rodzaj paliwa_Hybryda,Rodzaj paliwa_Hybryda Plug-in,...,Marka_Peugeot,Marka_Porsche,Marka_Renault,Marka_Seat,Marka_Skoda,Marka_Subaru,Marka_Suzuki,Marka_Toyota,Marka_Volkswagen,Marka_Volvo
0,327809,1,0,292,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,59000,118760,1969,254,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,80000,120000,1998,252,7,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,239000,22,0,1020,4,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,74900,5,1499,136,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Sprawdzanie ważności cech

In [16]:
korelacje = df.corr()['Cena'].sort_values(ascending=False)
print("\n--- TOP CECHY WPŁYWAJĄCE NA CENĘ ---")
print(korelacje.head(10)) # Co najbardziej podnosi cenę
print("\n--- CECHY OBNIŻAJĄCE CENĘ ---")
print(korelacje.tail(5))


--- TOP CECHY WPŁYWAJĄCE NA CENĘ ---
Cena                            1
Moc                             1
Pojemność skokowa               0
Marka_Mercedes-Benz             0
Rodzaj paliwa_Hybryda Plug-in   0
Marka_Porsche                   0
Marka_BMW                       0
Marka_Inne                      0
Marka_Land                      0
Rodzaj paliwa_Elektryczny       0
Name: Cena, dtype: float64

--- CECHY OBNIŻAJĄCE CENĘ ---
Rodzaj paliwa_Benzyna+LPG   -0
Marka_Opel                  -0
Skrzynia biegów_Manualna    -1
Przebieg                    -1
Wiek                        -1
Name: Cena, dtype: float64


Podział danych na X (cechy) i Y (czyli cel - Cena)

In [17]:
X = df.drop(columns=['Cena'])
y = df['Cena']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

prognozy_cen = model.predict(X_test)

mae = mean_absolute_error(y_test, prognozy_cen)
rmse = np.sqrt(mean_squared_error(y_test, prognozy_cen)) # Średni błąd kwadratowy spierwiastkowany
r2 = r2_score(y_test, prognozy_cen)

print("=== WYNIKI MODELU ===")
print(f"Średni Błąd (MAE): {mae:,.0f} PLN (O tyle średnio model myli się na aucie)")
print(f"Błąd Ważony (RMSE): {rmse:,.0f} PLN (Kara za drastyczne pomyłki w drogich autach)")
print(f"Skuteczność (R²): {r2 * 100:.1f} % (Jaki % zmienności cen ogarnia model)")

=== WYNIKI MODELU ===
Średni Błąd (MAE): 30,888 PLN (O tyle średnio model myli się na aucie)
Błąd Ważony (RMSE): 50,005 PLN (Kara za drastyczne pomyłki w drogich autach)
Skuteczność (R²): 70.5 % (Jaki % zmienności cen ogarnia model)


In [18]:
print("Test na 3 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': prognozy_cen,
    'Pomyłka (Błąd)': np.abs(y_test.values - prognozy_cen)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(3))

Test na 3 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000           112,237          54,237
1                      6500           -11,815          18,315
2                    116900           121,370           4,470


### Regresja wielomianowa

In [ ]:
kolumny_numeryczne = ['Przebieg', 'Pojemność skokowa', 'Moc', 'Wiek']

num_transformer = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, kolumny_numeryczne)
    ],
    remainder='passthrough'
)

model_poly = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

print("Trwa potęgowanie, skalowanie i trenowanie modelu...")
model_poly.fit(X_train, y_train)

prognozy_poly = model_poly.predict(X_test)

mae_poly = mean_absolute_error(y_test, prognozy_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, prognozy_poly))
r2_poly = r2_score(y_test, prognozy_poly)

print(f"Średni Błąd (MAE): {mae_poly:,.0f} PLN")
print(f"Błąd Ważony (RMSE): {rmse_poly:,.0f} PLN")
print(f"Skuteczność (R²): {r2_poly * 100:.1f} %")

Trwa potęgowanie, skalowanie i trenowanie modelu...
Średni Błąd (MAE): 21,842 PLN
Błąd Ważony (RMSE): 38,138 PLN
Skuteczność (R²): 82.8 %


In [21]:
print("Test na 3 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': prognozy_poly,
    'Pomyłka (Błąd)': np.abs(y_test.values - prognozy_poly)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(3))

Test na 3 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000            95,829          37,829
1                      6500             1,809           4,691
2                    116900           149,950          33,050
